# Reversed Headline Quality Evaluation

This notebook evaluates `reversed_headline` from `Sarcasm_Headlines_Dataset_v2_flipped.json` with the Hugging Face model `helinivan/english-sarcasm-detector`.

Workflow:
- Pull the repo into Colab if needed.
- Prompt for dataset upload at runtime.
- Run inference only on `reversed_headline`.
- Derive the expected label by flipping the original label (`1 - label` or `1 - is_sarcastic`).
- Compare detector predictions against the flipped label to estimate reversal quality.


The model card for `helinivan/english-sarcasm-detector` states that label `0` means not sarcastic and label `1` means sarcastic. Its example code also lowercases text and strips punctuation before tokenization, so the notebook follows that preprocessing.


In [ ]:
# Pull code, install dependencies

import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LINCHENYU2030S/CS4248_group_project.git"
REPO_ROOT = Path("/content/CS4248_group_project")
LOCAL_PROJECT_ROOT = Path("/content/CS4248_group_project")
LOCAL_PROJECT_EVAL_DIR = LOCAL_PROJECT_ROOT / "evaluation"
LOCAL_EVAL_DIR = Path("/content/evaluation")

current_dir = Path.cwd().resolve()
if current_dir.name == "evaluation":
    project_eval_dir = current_dir
elif current_dir.name == "notebooks" and (current_dir.parent / "evaluation").exists():
    project_eval_dir = current_dir.parent / "evaluation"
elif (current_dir / "evaluation").exists():
    project_eval_dir = current_dir / "evaluation"
elif LOCAL_PROJECT_EVAL_DIR.exists():
    project_eval_dir = LOCAL_PROJECT_EVAL_DIR
elif LOCAL_EVAL_DIR.exists():
    project_eval_dir = LOCAL_EVAL_DIR
elif "google.colab" in sys.modules:
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    project_eval_dir = REPO_ROOT / "evaluation"
else:
    raise RuntimeError(
        f"Could not locate the repo evaluation directory from: {current_dir}"
    )

os.chdir(project_eval_dir)
PROJECT_EVAL_DIR = Path.cwd().resolve()
if PROJECT_EVAL_DIR.name != "evaluation":
    raise RuntimeError(f"Expected to run inside the evaluation directory, got: {PROJECT_EVAL_DIR}")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if str(PROJECT_EVAL_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_EVAL_DIR))

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "transformers>=4.39.0",
        "accelerate>=0.28.0",
        "scikit-learn>=1.4.0",
        "pandas>=2.0.0",
        "matplotlib>=3.7.0",
        "ipywidgets>=8.1.0",
        "tqdm>=4.66.0",
    ],
    check=True,
)

print(f"Working directory: {PROJECT_EVAL_DIR}")


In [ ]:
# Upload dataset at run time

import shutil

DATASET_FILENAME = "Sarcasm_Headlines_Dataset_v2_flipped.json"
runtime_dataset_path = PROJECT_EVAL_DIR / DATASET_FILENAME

if runtime_dataset_path.exists():
    print(f"Dataset already available at: {runtime_dataset_path}")
elif "google.colab" in sys.modules:
    from google.colab import files

    print(f"Upload {DATASET_FILENAME} when the file picker opens.")
    uploaded = files.upload()

    if DATASET_FILENAME not in uploaded:
        raise FileNotFoundError(
            f"Expected {DATASET_FILENAME!r}. Uploaded files: {list(uploaded)}"
        )

    upload_candidates = [Path("/") / DATASET_FILENAME, Path.cwd() / DATASET_FILENAME]
    upload_source = next((path for path in upload_candidates if path.exists()), None)
    if upload_source is None:
        raise FileNotFoundError(
            f"Uploaded file not found in expected locations: {upload_candidates}"
        )

    shutil.move(str(upload_source), str(runtime_dataset_path))
    print(f"Dataset saved to: {runtime_dataset_path}")
else:
    raise FileNotFoundError(
        f"Dataset not found at {runtime_dataset_path}. Place the JSON file there before continuing."
    )

DATASET_PATH = runtime_dataset_path


In [ ]:
# Set seed, imports, and device

import random
import string

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from matplotlib import pyplot as plt
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, set_seed

pd.set_option("display.max_colwidth", 120)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE != "cuda":
    print("Warning: GPU is not enabled. Inference will be slower on CPU.")


In [ ]:
# Define model, columns, and output path

MODEL_NAME = "helinivan/english-sarcasm-detector"
TEXT_COLUMN = "reversed_headline"
FLIPPED_LABEL_COLUMN = "reversed_label"
MAX_LENGTH = 256
BATCH_SIZE = 64
SAMPLE_SIZE = None  # Set to an integer for a quicker debug run.
OUTPUT_PATH = PROJECT_EVAL_DIR / (Path(DATASET_FILENAME).stem + "_reversed_headline_eval.csv")

print(f"Dataset path: {DATASET_PATH}")
print(f"Output path: {OUTPUT_PATH}")


In [ ]:
# Load dataset and build flipped labels for reversed_headline

try:
    df = pd.read_json(DATASET_PATH, lines=True)
except ValueError:
    df = pd.read_json(DATASET_PATH)

if "label" in df.columns:
    original_label_column = "label"
elif "is_sarcastic" in df.columns:
    original_label_column = "is_sarcastic"
else:
    raise KeyError("Expected either 'label' or 'is_sarcastic' in the dataset.")

required_columns = {TEXT_COLUMN, original_label_column}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

df = df.dropna(subset=[TEXT_COLUMN]).copy()
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
df = df[df[TEXT_COLUMN] != ""].copy()
df[original_label_column] = df[original_label_column].astype(int)

invalid_labels = set(df[original_label_column].unique()) - {0, 1}
if invalid_labels:
    raise ValueError(
        f"Original label column must contain only 0/1 values. Found: {sorted(invalid_labels)}"
    )

df[FLIPPED_LABEL_COLUMN] = 1 - df[original_label_column]

if SAMPLE_SIZE is not None:
    df = df.head(SAMPLE_SIZE).copy()

preview_columns = [
    col for col in [original_label_column, FLIPPED_LABEL_COLUMN, "headline", TEXT_COLUMN] if col in df.columns
]
display(df[preview_columns].head())
print(f"Rows to evaluate: {len(df):,}")
print(df[FLIPPED_LABEL_COLUMN].value_counts().sort_index().rename(index={0: "not_sarcastic", 1: "sarcastic"}))


In [ ]:
# Load classifier and define inference helper

def preprocess_data(text: str) -> str:
    return text.lower().translate(str.maketrans("", "", string.punctuation)).strip()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

id2label = {0: "not_sarcastic", 1: "sarcastic"}
print(f"Loaded model: {MODEL_NAME}")
print(f"Model config labels: {getattr(model.config, 'id2label', {})}")

def predict_sarcasm(texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    predictions = []
    confidences = []
    sarcastic_probabilities = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Running inference"):
        batch_texts = [preprocess_data(text) for text in texts[start:start + batch_size]]
        tokenized = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(DEVICE)

        with torch.inference_mode():
            logits = model(**tokenized).logits
            probabilities = torch.softmax(logits, dim=-1)

        batch_confidences, batch_predictions = probabilities.max(dim=-1)
        predictions.extend(batch_predictions.cpu().tolist())
        confidences.extend(batch_confidences.cpu().tolist())
        sarcastic_probabilities.extend(probabilities[:, 1].cpu().tolist())

    return predictions, confidences, sarcastic_probabilities


In [ ]:
# Run inference on reversed_headline only

predictions, confidences, sarcastic_probabilities = predict_sarcasm(df[TEXT_COLUMN].tolist())

df["predicted_label"] = predictions
df["prediction_confidence"] = confidences
df["sarcastic_probability"] = sarcastic_probabilities
df["predicted_label_name"] = df["predicted_label"].map(id2label)
df["expected_label_name"] = df[FLIPPED_LABEL_COLUMN].map(id2label)

display(df[[TEXT_COLUMN, FLIPPED_LABEL_COLUMN, "predicted_label", "prediction_confidence"]].head())


In [ ]:
# Summarise prediction quality against flipped labels

y_true = df[FLIPPED_LABEL_COLUMN]
y_pred = df["predicted_label"]

summary = pd.DataFrame(
    [
        {
            "rows_evaluated": len(df),
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro"),
            "expected_sarcastic_rate": y_true.mean(),
            "predicted_sarcastic_rate": y_pred.mean(),
        }
    ]
)
display(summary)

report = pd.DataFrame(
    classification_report(
        y_true,
        y_pred,
        target_names=["not_sarcastic", "sarcastic"],
        output_dict=True,
        zero_division=0,
    )
).transpose()
display(report)


In [ ]:
# Visualise confusion matrix

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["not sarcastic", "sarcastic"],
)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title("Reversed headline prediction vs flipped label")
plt.show()


In [ ]:
# Inspect errors and save the full evaluation table

errors = df[df["predicted_label"] != df[FLIPPED_LABEL_COLUMN]].copy()
errors = errors.sort_values("prediction_confidence", ascending=False)

print(f"Misclassified reversed headlines: {len(errors):,}")
error_columns = [
    col
    for col in [
        "headline",
        TEXT_COLUMN,
        original_label_column,
        FLIPPED_LABEL_COLUMN,
        "predicted_label",
        "prediction_confidence",
        "sarcastic_probability",
    ]
    if col in errors.columns
]
display(errors[error_columns].head(25))

df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved full evaluation output to: {OUTPUT_PATH}")


In [ ]:
# Show example rows that were classified correctly and incorrectly

example_columns = [
    col
    for col in [
        "headline",
        TEXT_COLUMN,
        original_label_column,
        FLIPPED_LABEL_COLUMN,
        "predicted_label",
        "prediction_confidence",
        "sarcastic_probability",
    ]
    if col in df.columns
]

correct_examples = df[df["predicted_label"] == df[FLIPPED_LABEL_COLUMN]].copy()
incorrect_examples = df[df["predicted_label"] != df[FLIPPED_LABEL_COLUMN]].copy()

print(f"Correctly classified rows: {len(correct_examples):,}")
if len(correct_examples) > 0:
    display(correct_examples[example_columns].sample(n=min(5, len(correct_examples)), random_state=SEED))
else:
    print("No correctly classified rows available.")

print(f"Incorrectly classified rows: {len(incorrect_examples):,}")
if len(incorrect_examples) > 0:
    display(incorrect_examples[example_columns].sample(n=min(5, len(incorrect_examples)), random_state=SEED))
else:
    print("No incorrectly classified rows available.")


In [ ]:
# Save only rows where reversed_headline is correctly predicted by the classifier

FILTERED_OUTPUT_PATH = PROJECT_EVAL_DIR / "nonsarcastic_to_sarcastic_filtered.json"
added_columns = {
    FLIPPED_LABEL_COLUMN,
    "predicted_label",
    "prediction_confidence",
    "sarcastic_probability",
    "predicted_label_name",
    "expected_label_name",
}
original_columns = [column for column in df.columns if column not in added_columns]

filtered_df = df[df["predicted_label"] == df[FLIPPED_LABEL_COLUMN]].copy()

with FILTERED_OUTPUT_PATH.open("w", encoding="utf-8") as fout:
    for record in filtered_df[original_columns].to_dict(orient="records"):
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Kept {len(filtered_df):,} / {len(df):,} rows where reversed_headline was correctly predicted.")
print(f"Saved filtered dataset to: {FILTERED_OUTPUT_PATH}")
display(filtered_df[original_columns].head(10))
